# Consultas Compostas: UNION, Subconsultas, IN e EXISTS

---

## Contexto: perguntas que exigem mais de uma query

O gerente comercial quer respostas que uma query simples não consegue dar:

- *"Quais pedidos precisam de atenção recentes OU de alto valor?"*
- *"Quais clientes compraram produtos de Eletrônicos?"*
- *"Quais produtos nunca foram vendidos?"*
- *"Quais categorias têm mais de 3 produtos cadastrados?"*

Para isso existem as **consultas compostas**: o resultado de uma query alimenta outra.

Nesta aula você vai aprender:
- `UNION` / `UNION ALL` — combinar conjuntos de resultados
- `subquery()` — usar uma query como etapa intermediária
- `IN` / `NOT IN` — filtrar por pertencimento
- `EXISTS` / `NOT EXISTS` — filtrar por existência

---

## 1. Configuração

In [ ]:
# Bibliotecas necessárias
from pathlib import Path
from datetime import datetime, timedelta
from decimal import Decimal
from sqlalchemy import (
    create_engine, select, and_, or_, func, inspect,
    String, DateTime, Numeric, Integer, Boolean, ForeignKey, CheckConstraint, Index,
    union, union_all, exists
)
from sqlalchemy.orm import (
    DeclarativeBase, Mapped, mapped_column, relationship, Session,
    joinedload, selectinload
)
from models import Base, Cliente, Produto, ItemPedido, Pedido

# Cria o diretório do banco se não existir
Path("../database").mkdir(exist_ok=True)

# Cria a engine para conectar ao banco SQLite
engine = create_engine("sqlite:///../database/datavendas.db")
# Confirmação visual de que a conexão/engine foi configurada com sucesso
print("Conexão OK ✅")

## 2. Construindo queries em etapas

Antes de entrar nas consultas compostas, um padrão útil: o `select()` pode ser **construído aos poucos**.  
Como ele só vai ao banco quando você chama `execute()`, você pode ir adicionando filtros conforme a lógica exige:

In [ ]:
# código:

## 3. UNION — combinando dois conjuntos

Use `UNION` para juntar resultados de duas queries em uma lista única.  
**Regra:** as duas queries precisam retornar as **mesmas colunas** (mesmos tipos, mesma ordem).

**Caso de uso:** lista de pedidos prioritários para auditoria: recentes **OU** de alto valor.

In [ ]:
# código

| | Remove duplicatas | Velocidade | Quando usar |
|---|---|---|---|
| `union()` | ✅ Sim | Mais lento | Cada registro deve aparecer uma única vez |
| `union_all()` | ❌ Não | Mais rápido | Quando você quer contar ocorrências |

## 4. Subconsultas — query dentro de query

O `.subquery()` transforma uma query em um bloco reutilizável dentro de outra query.

In [ ]:
# código

**O padrão é sempre este:**
```
1. Montar a query intermediária  →  select(...).where(...)
2. Converter em subquery          →  .subquery()
3. Usar na query principal        →  .join(subq, ...) ou .where(col.in_(subq))
```

## 4. IN — filtrar por pertencimento

In [ ]:
# código:

## 5. EXISTS — filtrar por existência

`EXISTS` responde: *"existe ao menos um registro que satisfaça essa condição?"*  
Ele apenas **verifica** não traz dados, o que costuma ser mais rápido para verificações.

In [ ]:
# código:

| | `IN` | `EXISTS` |
|---|---|---|
| Pergunta típica | "O ID está nessa lista?" | "Existe algum registro relacionado?" |
| Negação | `.notin_()` | `~exists()` |
| Melhor para | Listas pequenas/médias | Verificar relacionamentos |

## 7. HAVING — filtrando grupos

---

## Exercício: Usando IA para isso

Consultas compostas são um ótimo caso de uso para IA — especialmente para decompor perguntas complexas.

**Prompt para decompor uma pergunta em query:**
```
Com os modelos SQLAlchemy abaixo, escreva a query para responder:
"Quais clientes que compraram na última semana NÃO compraram nenhum
produto da categoria 'Promoção'?"
Use subconsultas com IN ou NOT EXISTS conforme apropriado.
Explique cada etapa.

[modelos ORM]
```
---

### Resposta:Código gerado pelo Copilot

In [ ]:
from sqlalchemy import select, and_, not_
from datetime import datetime, timedelta

# Calcula a data de 7 dias atrás
data_corte = datetime.now() - timedelta(days=7)

with Session(engine) as session:
    # ===== ETAPA 1: IDs de clientes que compraram na última semana =====
    # Busca pedidos dos últimos 7 dias e extrai os IDs dos clientes
    ids_clientes_semana = (
        select(Pedido.cliente_id)
        .where(Pedido.data_pedido >= data_corte)
        .distinct()
    )
    
    # ===== ETAPA 2: IDs de clientes que compraram produtos de 'Promoção' =====
    # Caminho: Produto (categoria='Promoção') → ItemPedido → Pedido → Cliente
    ids_clientes_promocao = (
        select(Pedido.cliente_id)
        .join(Pedido.itens)  # JOIN com ItemPedido
        .join(ItemPedido.produto)  # JOIN com Produto
        .where(Produto.categoria == "Promoção")
        .distinct()
    )
    
    # ===== ETAPA 3: Clientes que compraram na última semana MAS NÃO em 'Promoção' =====
    # Usa NOT IN: clientes da semana que NÃO estão na lista de compradores de promoção
    stmt = (
        select(Cliente)
        .where(
            and_(
                Cliente.id.in_(ids_clientes_semana),  # Comprou na última semana
                ~Cliente.id.in_(ids_clientes_promocao)  # MAS não comprou Promoção
            )
        )
        .order_by(Cliente.nome)
    )
    
    clientes = session.scalars(stmt).all()
    
    print(f"Clientes que compraram na última semana, excluindo 'Promoção': {len(clientes)}")
    for c in clientes:
        print(f"  {c.nome} — {c.cidade}/{c.estado}")

# ===== ALTERNATIVA COM NOT EXISTS =====
# Mais eficiente para grandes volumes (não materializa listas)

with Session(engine) as session:
    # Subquery correlacionada: verifica se cliente comprou na semana mas NÃO em Promoção
    has_compra_semana = (
        select(Pedido.id)
        .where(
            and_(
                Pedido.cliente_id == Cliente.id,  # Correlação com query externa
                Pedido.data_pedido >= data_corte  # Na última semana
            )
        )
    )
    
    has_promocao = (
        select(ItemPedido.id)
        .join(ItemPedido.pedido)  # JOIN com Pedido
        .where(
            and_(
                Pedido.cliente_id == Cliente.id,  # Correlação com query externa
                ItemPedido.produto_id == Produto.id,
                Produto.categoria == "Promoção"
            )
        )
    )
    
    stmt = (
        select(Cliente)
        .where(
            and_(
                exists(has_compra_semana),  # Comprou na semana
                ~exists(has_promocao)  # MAS não tem Promoção
            )
        )
        .order_by(Cliente.nome)
    )
    
    clientes = session.scalars(stmt).all()
    print(f"\nCom NOT EXISTS (alternativa): {len(clientes)} clientes")